In [2]:
from typing import TypedDict

from langgraph.graph import StateGraph,START,END
from langgraph.types import RetryPolicy
from loguru import  logger
from requests import HTTPError


#1. 声明状态
class EmptyState(TypedDict):
    pass
#2. 声明节点
def node_a(state:EmptyState) -> EmptyState:
    logger.info("node a正在运行")
    raise HTTPError

#3. 构建图
builder = StateGraph(state_schema = EmptyState)
builder.add_node(
    "node_a",
    node_a,
    retry_policy=RetryPolicy(
        max_attempts=3,
        jitter = False
    ))

builder.add_edge(START,"node_a")
builder.add_edge("node_a",END)
graph = builder.compile()

try:
    graph.invoke({})
except HTTPError as e:
    logger.info("重试次数耗尽:{}",e)



2026-07-06 11:23:03.028 | INFO     | __main__:node_a:14 - node a正在运行
2026-07-06 11:23:03.529 | INFO     | __main__:node_a:14 - node a正在运行
2026-07-06 11:23:04.530 | INFO     | __main__:node_a:14 - node a正在运行
2026-07-06 11:23:04.531 | INFO     | __main__:<module>:34 - 重试次数耗尽:
